# 2011 SOXX 披露篮子集中度来源核验

原文、版式、预算和停止条件见 `plan.json`。只处理四份已核验原始报表，零市场结果/拟合/账户。


In [1]:
import json,hashlib,re
from pathlib import Path
from bs4 import BeautifulSoup
from datetime import datetime,timezone
root=Path.cwd();card=root/'research/stage1_soxx_2011_reported_concentration_source_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['parent_source_summary_sha256']==sha(root/'research/stage1_soxx_legacy_basket_history_20261023/summary.json')
claim=card/'attempt_started.json';out=card/'source_result.json'
assert not claim.exists() and not out.exists()
claim.write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'raw_report_parses_started':4},indent=2)+'\n')
integer=re.compile(r'^[0-9][0-9,]*$')
results=[]
for item in plan['reports']:
 evidence=root/item['source_identity_receipt']
 assert sha(evidence)==item['source_identity_receipt_sha256']
 e=json.loads(evidence.read_text())
 nas=e.get('nas_evidence',{}).get('sha256',e.get('source_NAS_archive_sha256'))
 readback=e.get('nas_scoped_restore_verified',e.get('source_NAS_scoped_restore_verified'))
 assert nas==item['source_NAS_archive_sha256'] and readback is True
 raw=root/item['raw_path'];payload=raw.read_bytes()
 assert len(payload)==item['raw_bytes'] and hashlib.sha256(payload).hexdigest()==item['raw_sha256']
 a,b=item['bounded_section_byte_range'];section=payload[a:b]
 assert hashlib.sha256(section).hexdigest()==item['bounded_section_sha256']
 rows=BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser').find_all('tr')
 parsed=[];ambiguous=[];start=False;total_label=False;printed=None
 for row in rows:
  cells=[x.get_text(' ',strip=True).replace('\xa0',' ').strip() for x in row.find_all(['td','th'],recursive=False)]
  nonempty=[x for x in cells if x and x!='$']
  if not nonempty:continue
  name=nonempty[0];upper=name.upper()
  nums=[int(x.replace(',','')) for x in nonempty[1:] if integer.fullmatch(x)]
  if not start:
   if 'COMMON STOCKS' in upper and 'TOTAL' not in upper:start=True
   continue
  if 'TOTAL COMMON STOCKS' in upper:
   total_label=True
   if nums:printed=nums[-1];break
   continue
  if total_label:
   if nums:printed=nums[-1]
   break
  if upper==name and ('—' in name or '–' in name or 'COMMON STOCKS' in upper):continue
  if len(nums)>=2 and any(ch.isalpha() for ch in name):
   parsed.append({'security_original':name,'shares_original':nums[0],'fair_value_usd_original':nums[-1]})
  elif any(ch.isalpha() for ch in name) and nums:
   ambiguous.append({'row_text':nonempty[:5],'numeric_tokens':nums})
 values=[x['fair_value_usd_original'] for x in parsed];total=sum(values)
 for x in parsed:x['reported_equity_weight']=x['fair_value_usd_original']/total if total else None
 checks={'common_stocks_heading_found':start,'total_label_found':total_label,'printed_total_found':printed is not None,
  'row_count_20_to_40':20<=len(parsed)<=40,'all_positive':all(x['shares_original']>0 and x['fair_value_usd_original']>0 for x in parsed),
  'distinct_names':len({x['security_original'] for x in parsed})==len(parsed),
  'total_closure_within_1_usd':printed is not None and abs(total-printed)<=1,'ambiguous_rows_zero':len(ambiguous)==0}
 passed=all(checks.values())
 ranked=sorted(parsed,key=lambda x:x['fair_value_usd_original'],reverse=True)
 results.append({'report_date':item['report_date'],'form':item['form'],'accession':item['accession'],'accepted_at_utc':item['accepted_at_utc'],
  'raw_sha256':item['raw_sha256'],'section_sha256':item['bounded_section_sha256'],'source_NAS_archive_sha256':nas,
  'source_NAS_scoped_restore_verified':True,'status':'numeric_source_closure_passed' if passed else 'numeric_source_unqualified',
  'parsed_equity_rows':len(parsed),'equities':parsed,'ambiguous_rows':ambiguous,
  'parsed_fair_value_total_usd':total,'printed_total_common_stocks_usd':printed,
  'total_closure_difference_usd':total-printed if printed is not None else None,'checks':checks,
  'top5_disclosed_equity_value_share':sum(x['reported_equity_weight'] for x in ranked[:5]) if total else None,
  'herfindahl_sum_squared_disclosed_equity_weights':sum(x['reported_equity_weight']**2 for x in parsed) if total else None})
result={'status':'four_2011_original_SOXX_reported_value_sources_closed' if all(x['status']=='numeric_source_closure_passed' for x in results) else '2011_original_SOXX_reported_value_sources_partial_or_unqualified',
 'plan_sha256':sha(card/'plan.json'),'reports':results,'reports_attempted':len(results),
 'reports_with_numeric_closure':sum(x['status']=='numeric_source_closure_passed' for x in results),
 'full_2011_2022_training_coverage':False,'held_2023_2026_coverage':False,
 'HTTP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'scientific_boundary':'Four source values alone do not establish incremental information or a 0bp account.'}
out.write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'per_report':[(x['report_date'],x['parsed_equity_rows'],x['total_closure_difference_usd'],x['status']) for x in results]})


{'status': 'four_2011_original_SOXX_reported_value_sources_closed', 'per_report': [('2011-01-31', 30, 0, 'numeric_source_closure_passed'), ('2011-04-30', 30, 0, 'numeric_source_closure_passed'), ('2011-07-31', 30, 0, 'numeric_source_closure_passed'), ('2011-10-31', 30, 0, 'numeric_source_closure_passed')]}


## 中文结论

执行后见 `summary.json` 与 `conclusion.md`；来源闭合不代表预测力或 SOXX 超额。
